In [5]:
import pandas as pd
from pyprojroot import here
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from datetime import time
from marginal_emissions import logger
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import time

path = here() / "data" / "raw" / "other" / "smard"

In [6]:
res_loads = {
    '50Hertz': pd.read_csv(f'{path}/Realisierter_Stromverbrauch_50Hertz_202301010000_202501010000_Viertelstunde.csv', sep=';'),
    'Amprion': pd.read_csv(f'{path}/Realisierter_Stromverbrauch_Amprion_202301010000_202501010000_Viertelstunde.csv', sep=';'),
    'TenneT': pd.read_csv(f'{path}/Realisierter_Stromverbrauch_TenneT_202301010000_202501010000_Viertelstunde.csv', sep=';'),
    'TransnetBW': pd.read_csv(f'{path}/Realisierter_Stromverbrauch_TransnetBW_202301010000_202501010000_Viertelstunde.csv', sep=';')
}

for area, df in res_loads.items():
    # Select only the index column and residual load
    df = df.loc[:, ['Datum von', 'Residuallast [MWh] Originalauflösungen']].rename(
        columns={
            'Datum von': 'datetime',
            'Residuallast [MWh] Originalauflösungen': 'Residual Load'
        }
    )
    # Transform types
    df['datetime'] = pd.to_datetime(df['datetime'], format="%d.%m.%Y %H:%M")
    df.set_index('datetime', inplace=True)
    if not df.index.dtype == 'datetime64[ns, UTC]':
        try:
            df.index = pd.to_datetime(df.index, format='ISO8601')
            df.index = df.index.tz_localize('Europe/Berlin', ambiguous='infer')
            df.index = df.index.tz_convert('UTC')
            df.sort_index(inplace=True)
        except ValueError:
            print(f"Failed to set index to datetime for {area}")

    if not pd.api.types.is_float_dtype(df['Residual Load']):
        df['Residual Load'] = (
            df['Residual Load']
            .astype(str)
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        df['Residual Load'] = pd.to_numeric(df['Residual Load'], errors='coerce')

    res_loads[area] = df

In [7]:
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import time

def plot_residual_load_derivatives(res_loads):
    """
    Plots the average daily profile of residual loads, their 1st and 2nd derivatives
    in a 2x2 subplot layout. Marks local extrema and shades the four TVTP phases.
    Includes vertical lines for Geske & Green (2020) phases without inline text.
    Extracts these extrema into a table and saves them as a CSV.
    Saves the plot as a PNG file.
    """
    with plt.style.context('default'):
        fig, axes = plt.subplots(2, 2, figsize=(16, 9))
        axes = axes.flatten()

        # Colors for the different lines in each subplot
        c_res = '#1f77b4'  # Blue for Residual Load
        c_d1 = '#ff7f0e'   # Orange for 1st Derivative
        c_d2 = '#2ca02c'   # Green for 2nd Derivative

        # Empirical phases derived from data (Start time, End time, Label, Color)
        phases = [
            ('00:00:00', '06:00:00', 'Phase 1: Overnight Trough', '#7f7f7f'),
            ('06:00:00', '10:00:00', 'Phase 2: Morning Peak', '#d62728'),
            ('10:00:00', '16:00:00', 'Phase 3: Solar Trough', '#bcbd22'),
            ('16:00:00', '23:59:59', 'Phase 4: Evening Peak', '#9467bd')
        ]

        # Theoretical times from Geske & Green (2020)
        gg_times = [time(0, 0), time(7, 0), time(9, 0), time(15, 0), time(17, 0)]

        handles_collected = False
        global_handles = []
        global_labels = []

        extrema_list = []

        for i, (area, df) in enumerate(res_loads.items()):
            ax1 = axes[i]
            ax2 = ax1.twinx()

            # 1. Prepare Data
            df_local = df.copy()
            df_local.index = df_local.index.tz_convert('Europe/Berlin')

            daily_avg = df_local.groupby(df_local.index.time).mean()
            d1 = daily_avg['Residual Load'].diff()
            d2 = d1.diff()

            plot_times = pd.to_datetime(daily_avg.index.astype(str), format='%H:%M:%S')

            # 2. Find local Minima and Maxima on the Residual Load curve
            y = daily_avg['Residual Load']
            is_max = (y > y.shift(1)) & (y > y.shift(-1))
            is_min = (y < y.shift(1)) & (y < y.shift(-1))

            # Table extraction
            for time_idx in y[is_max].index:
                extrema_list.append({
                    'TSO': area, 'Extremum': 'Maximum', 'Time': time_idx.strftime('%H:%M'),
                    'Residual Load [MWh]': round(y[time_idx], 2),
                    '1st Derivative': round(d1[time_idx], 2) if pd.notnull(d1[time_idx]) else None,
                    '2nd Derivative': round(d2[time_idx], 2) if pd.notnull(d2[time_idx]) else None
                })
            for time_idx in y[is_min].index:
                extrema_list.append({
                    'TSO': area, 'Extremum': 'Minimum', 'Time': time_idx.strftime('%H:%M'),
                    'Residual Load [MWh]': round(y[time_idx], 2),
                    '1st Derivative': round(d1[time_idx], 2) if pd.notnull(d1[time_idx]) else None,
                    '2nd Derivative': round(d2[time_idx], 2) if pd.notnull(d2[time_idx]) else None
                })

            # 3. Add background shading for the empirical phases
            for start_str, end_str, label, color in phases:
                start_t = pd.to_datetime(start_str, format='%H:%M:%S')
                end_t = pd.to_datetime(end_str, format='%H:%M:%S')
                ax1.axvspan(start_t, end_t, color=color, alpha=0.15,
                            label=label if not handles_collected else None, zorder=1)

            # 4. Add Geske & Green (2020) Lines
            for j, t in enumerate(gg_times):
                plot_t = pd.to_datetime(str(t), format='%H:%M:%S')
                ax1.axvline(x=plot_t, color='black', linestyle=':', linewidth=1.5, alpha=0.6,
                            label='Geske & Green (2020)' if j == 0 and not handles_collected else None, zorder=2)

            # 5. Plot Lines
            line_res, = ax1.plot(plot_times, y, label='Residual Load', color=c_res, linewidth=2.5, zorder=3)
            line_d1, = ax2.plot(plot_times, d1, label='1st Derivative', color=c_d1, linestyle='--', linewidth=1.5, alpha=0.85, zorder=3)
            line_d2, = ax2.plot(plot_times, d2, label='2nd Derivative', color=c_d2, linestyle='-.', linewidth=1.5, alpha=0.85, zorder=3)

            # 6. Mark Minima and Maxima
            max_times = plot_times[is_max]
            max_y = y[is_max]
            max_d2 = d2[is_max]

            min_times = plot_times[is_min]
            min_y = y[is_min]
            min_d2 = d2[is_min]

            # Markers on Residual Load
            ax1.scatter(max_times, max_y, color='red', s=60, edgecolors='black', zorder=4, label='Local Maxima' if not handles_collected else None)
            ax1.scatter(min_times, min_y, color='blue', s=60, edgecolors='black', zorder=4, label='Local Minima' if not handles_collected else None)

            # Markers on 2nd Derivative
            ax2.scatter(max_times, max_d2, color='red', s=40, marker='X', edgecolors='black', zorder=4)
            ax2.scatter(min_times, min_d2, color='blue', s=40, marker='X', edgecolors='black', zorder=4)

            # 7. Formatting Subplot
            ax1.set_title(f"{area}", fontsize=13, fontweight='bold')
            ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

            # X-Axis Label added here
            ax1.set_xlabel("Time", fontsize=10)

            ax1.set_ylabel("Residual Load [MWh]", fontsize=10)
            ax2.set_ylabel("Derivatives", fontsize=10)
            ax1.set_xlim([pd.to_datetime('00:00:00', format='%H:%M:%S'), pd.to_datetime('23:59:59', format='%H:%M:%S')])
            ax1.grid(True, alpha=0.3, zorder=1)

            # Collect handles
            if not handles_collected:
                h1, l1 = ax1.get_legend_handles_labels()
                h2, l2 = ax2.get_legend_handles_labels()
                for handle, label in zip(h1 + h2, l1 + l2):
                    if label not in global_labels and label is not None:
                        global_handles.append(handle)
                        global_labels.append(label)
                handles_collected = True

        # --- AB HIER IST DER FIX FÜR DIE LEGENDE ---

        # 1. Wir reservieren die oberen 12% des Bildes (rect=[left, bottom, right, top]) exklusiv für die Legende.
        # So können die Subplots niemals in die Legende hineinwachsen.
        plt.tight_layout(rect=[0, 0, 1, 0.88])

        # 2. Wir setzen die Legende exakt in den geschaffenen Freiraum.
        # loc='lower center' bedeutet, der untere Rand der Legende wird an die Y-Koordinate 0.89 gesetzt.
        fig.legend(global_handles, global_labels, loc='lower center',
                   bbox_to_anchor=(0.5, 0.89), ncol=5, fontsize=10, frameon=True)

        # Save output
        df_extrema = pd.DataFrame(extrema_list)
        if not df_extrema.empty:
            df_extrema = df_extrema.sort_values(by=['TSO', 'Time']).reset_index(drop=True)
            print("\n--- Residual Load Extrema (Minima & Maxima) ---")
            print(df_extrema.to_markdown(index=False))

        try:
            save_dir = here() / "results" / "autocorrelation"
            os.makedirs(save_dir, exist_ok=True)

            filename_plot = save_dir / "residual_load_derivatives_2x2.png"
            filename_csv = save_dir / "residual_load_extrema.csv"

            # bbox_inches='tight' sorgt beim Speichern dafür, dass nichts abgeschnitten wird
            plt.savefig(filename_plot, dpi=300, bbox_inches='tight')
            if not df_extrema.empty:
                df_extrema.to_csv(filename_csv, index=False, sep=';', decimal=',')

            try:
                logger.info(f"Derivatives plot saved to {filename_plot}")
            except NameError:
                pass
        except Exception as e:
            try:
                logger.error(f"Failed to save output files: {e}. Continuing...")
            except NameError:
                pass
        finally:
            plt.close(fig)

In [8]:
plot_residual_load_derivatives(res_loads=res_loads)


--- Residual Load Extrema (Minima & Maxima) ---
| TSO        | Extremum   | Time   |   Residual Load [MWh] |   1st Derivative |   2nd Derivative |
|:-----------|:-----------|:-------|----------------------:|-----------------:|-----------------:|
| 50Hertz    | Minimum    | 02:15  |               1104.05 |            -3.69 |             6.64 |
| 50Hertz    | Maximum    | 07:15  |               1541.2  |             3.29 |           -28.68 |
| 50Hertz    | Minimum    | 13:30  |                526.64 |            -0.58 |             3.77 |
| 50Hertz    | Maximum    | 19:30  |               1845.84 |            10.81 |            -7.26 |
| Amprion    | Minimum    | 02:15  |               2965.15 |            -6    |            24.78 |
| Amprion    | Maximum    | 08:30  |               4066.67 |             9.55 |           -19.36 |
| Amprion    | Minimum    | 14:00  |               3382.64 |           -11.28 |            23.92 |
| Amprion    | Maximum    | 19:00  |               4297.24 |